Tourist Mode - Retrieval MVP
Using rule based filters and cosine similarity to find matching restaurants

In [ ]:
import pandas as pd
import ast
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity


class TouristRetrieval:
    def __init__(self, file_path):
        self.df = pd.read_csv(file_path)
        self.df['price_range'] = self.df['attributes'].apply(self.get_price)
        self.df['categories'] = self.df['categories'].fillna('').str.lower()
        self.df = self.df[self.df['is_open']==1]
        self.vectorizer = CountVectorizer(stop_words=['food', 'restaurants'])
        self.vectorizer.fit(self.df['categories'])

    def get_price(self, attributes_str):
        if pd.isna(attributes_str) or attributes_str in ["{}", "None"]:
            return None 
        try:
            attr_dict = ast.literal_eval(attributes_str)
            price = attr_dict.get('RestaurantsPriceRange2')
            if price is None or str(price).strip().lower() in ['none', 'nan', '']:
                return None
            return int(price)
        except:
            return None
        
    def get_recommendations(self, cuisine_query, min_stars=3, max_price=4, top_k=5):
        query_str = f"stars >= {min_stars}"
        
        if max_price is not None and max_price < 4:
            query_str += f" and price_range <= {max_price}"
            
        filtered_df = self.df.query(query_str).copy()
        if filtered_df.empty:
            return "No matches found for those filters."

        subset_matrix = self.vectorizer.transform(filtered_df['categories'])
        query_vec = self.vectorizer.transform([cuisine_query.lower()])
        
        scores = cosine_similarity(subset_matrix, query_vec).flatten()
        filtered_df['match_score'] = scores

        results = filtered_df.sort_values(by=['match_score', 'stars'], ascending=False)

        return results.head(top_k)[['match_score', 'name', 'stars', 'price_range', 'categories']]
engine = TouristRetrieval("../data/cleaned_philly_restaurants.csv")

Test 1: tourist wants Japanese restaurants that are cheap and have a high star rating

In [7]:
print(engine.get_recommendations("Japanese", min_stars=4.5, max_price=2))


      match_score                  name  stars  price_range  \
600      1.000000            Poke Burri    4.5          2.0   
2039     0.707107  Maple Japanese Ramen    4.5          2.0   
4244     0.707107     Hajimaru Fishtown    4.5          2.0   
5484     0.577350          Hikari Sushi    5.0          2.0   
3561     0.577350            Fat Salmon    4.5          2.0   

                             categories  
600               restaurants, japanese  
2039       ramen, restaurants, japanese  
4244       restaurants, japanese, ramen  
5484  restaurants, sushi bars, japanese  
3561  restaurants, sushi bars, japanese  


Test 2: tourist wants a greek restaurant with at least 4 stars and they don't care about price

In [8]:
print(engine.get_recommendations("greek", min_stars=4))

      match_score                   name  stars  price_range  \
4589     1.000000  South Street Souvlaki    4.0          2.0   
273      0.707107        Zorba's Taverna    4.5          2.0   
5630     0.707107   Angelo's Pizza House    4.0          2.0   
3503     0.577350  Kostas Bar Restaurant    4.0          2.0   
5328     0.577350                  Estia    4.0          3.0   

                                      categories  
4589                          greek, restaurants  
273            restaurants, greek, mediterranean  
5630                   restaurants, greek, pizza  
3503         restaurants, bars, greek, nightlife  
5328  restaurants, greek, mediterranean, seafood  
